In [15]:
from pathlib import Path
import numpy as np
from astroquery.mast import Observations
from astropy.io import fits
from matplotlib import pyplot as plt

DOWNLOAD_DIR = Path.cwd() / "mast_miri_darks"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)


In [16]:
# MIRI Imager darks are calibration exposures with the OPAQUE filter
# (target_name is usually UNKNOWN, not "DARK").
obs_table = Observations.query_criteria(
    obs_collection="JWST",
    instrument_name="MIRI/IMAGE",
    intentType="calibration",
    filters="OPAQUE",
)

print(f"Found {len(obs_table)} MIRI/IMAGE OPAQUE calibration observations")
obs_table[["obs_id", "instrument_name", "filters", "t_exptime", "proposal_id"]][:10]


Found 439 MIRI/IMAGE OPAQUE calibration observations


obs_id,instrument_name,filters,t_exptime,proposal_id
str34,str10,str6,float64,str5
jw07928082001_02101_00001_mirimage,MIRI/IMAGE,OPAQUE,149.276,7928
jw07928037001_02101_00001_mirimage,MIRI/IMAGE,OPAQUE,375.598,7928
jw07928056001_02102_00001_mirimage,MIRI/IMAGE,OPAQUE,1218.386,7928
jw07945013001_04201_00001_mirimage,MIRI/IMAGE,OPAQUE,1390.295,7945
jw07945006001_04201_00001_mirimage,MIRI/IMAGE,OPAQUE,1390.295,7945
jw04482014001_02102_00001_mirimage,MIRI/IMAGE,OPAQUE,1218.386,4482
jw04482008001_02101_00001_mirimage,MIRI/IMAGE,OPAQUE,255.7,4482
jw04482003001_02101_00001_mirimage,MIRI/IMAGE,OPAQUE,1390.295,4482
jw06610009001_02102_00001_mirimage,MIRI/IMAGE,OPAQUE,1218.386,6610


In [17]:
# Download one calibrated MIRI DARK product (*_dark.fits)
if len(obs_table) == 0:
    raise RuntimeError("No MIRI/IMAGE OPAQUE calibration observations found.")

products = Observations.get_product_list(obs_table[0])
dark_products = Observations.filter_products(
    products,
    productSubGroupDescription=["DARK"],
)
if len(dark_products) == 0:
    raise RuntimeError("No DARK products found for this MIRI observation.")

print(dark_products["productFilename"][:5])
manifest = Observations.download_products(
    dark_products[:1],
    download_dir=str(DOWNLOAD_DIR),
)
print(manifest)

dark_path = Path(manifest["Local Path"][0])
print("Downloaded MIRI dark frame:", dark_path)


              productFilename               
--------------------------------------------
jw07928082001_02101_00001_mirimage_dark.fits
INFO: Found cached file /Users/eckhartspalding/Documents/git.repos/life_detectors/dev_notebooks/mast_miri_darks/mastDownload/JWST/jw07928082001_02101_00001_mirimage/jw07928082001_02101_00001_mirimage_dark.fits with expected size 130708800. [astroquery.query]
                                                                                        Local Path                                                                                        ...
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- ...
/Users/eckhartspalding/Documents/git.repos/life_detectors/dev_notebooks/mast_miri_darks/mastDownload/JWST/jw07928082001_02101_00001_mirimage/jw07928082001_02101_00001_mirimage_dark.fits ...
Downloaded MIRI dark frame: /Users/e

In [18]:
with fits.open(dark_path) as hdul:
    sci = hdul["SCI"].data
    hdr = hdul["SCI"].header
    print(sci.shape)   # expect (5, 250, 128, 136) = (nints, ngroups, ny, nx)
    print(hdr.get("BUNIT"))


(5, 250, 128, 136)
DN


In [19]:
# convert ramp -> rate

# seconds between groups
tgroup = hdr.get("TGROUP") or hdul[0].header.get("TGROUP")

# Average adjacent-group differences over ints & groups → DN (or e-) per second
# sci: (nints, ngroups, ny, nx)
dcounts = np.diff(sci, axis=1)          # (nints, ngroups-1, ny, nx)
rate_2d = np.nanmedian(dcounts, axis=(0, 1)) / tgroup   # (ny, nx)

In [20]:
# Save the computed rate_2d as a FITS file
rate_fits_path = dark_path.parent / (dark_path.stem + "_rate2d.fits")
hdu = fits.PrimaryHDU(rate_2d)
hdu.writeto(rate_fits_path, overwrite=True)
print(f"Saved rate_2d to {rate_fits_path}")

Saved rate_2d to /Users/eckhartspalding/Documents/git.repos/life_detectors/dev_notebooks/mast_miri_darks/mastDownload/JWST/jw07928082001_02101_00001_mirimage/jw07928082001_02101_00001_mirimage_dark_rate2d.fits


In [21]:
# CRDS master dark -> 2D rate via OLS fit of full ramps
from pathlib import Path
import numpy as np
from astropy.io import fits

CRDS_DARK = Path.home() / "crds_cache/references/jwst/miri/jwst_miri_dark_0113.fits"
# MIRI Imager FULL FASTR1: one frame per group → TGROUP = TFRAME = 2.775 s
# (CRDS dark header has no TGROUP/TFRAME; see JWST MIRI readout docs)
TGROUP_FULL_FASTR1 = 2.775  # seconds

OUT_RATE = CRDS_DARK.with_name(CRDS_DARK.stem + "_rate2d.fits")
CHUNK_ROWS = 64


In [22]:
def ols_rate_chunk(sci_chunk, t_c, denom):
    """OLS slope per pixel for sci_chunk (nints, ngroups, ny, nx).

    Returns mean rate over integrations, shape (ny, nx), in DN/s.
    """
    # center counts along group axis for numerical stability
    yc = sci_chunk - sci_chunk.mean(axis=1, keepdims=True)
    # slope = sum_g (y_c * t_c) / sum(t_c^2)
    rates = np.einsum("igyx,g->iyx", yc, t_c, optimize=True) / denom
    return rates.mean(axis=0)


with fits.open(CRDS_DARK, memmap=True) as hdul:
    sci = hdul["SCI"].data  # (nints, ngroups, ny, nx)
    nints, ngroups, ny, nx = sci.shape
    assert nints == 2 and ngroups == 360

    t = np.arange(ngroups, dtype=np.float64) * TGROUP_FULL_FASTR1
    t_c = t - t.mean()
    denom = float(np.dot(t_c, t_c))

    rate_2d = np.empty((ny, nx), dtype=np.float32)
    for y0 in range(0, ny, CHUNK_ROWS):
        y1 = min(y0 + CHUNK_ROWS, ny)
        chunk = np.asarray(sci[:, :, y0:y1, :], dtype=np.float64)
        rate_2d[y0:y1, :] = ols_rate_chunk(chunk, t_c, denom).astype(np.float32)
        print(f"fitted rows {y0}:{y1}", flush=True)

print(
    "rate_2d DN/s: "
    f"min={rate_2d.min():.4g} max={rate_2d.max():.4g} "
    f"median={np.nanmedian(rate_2d):.4g} mean={np.nanmean(rate_2d):.4g}"
)

hdr = fits.Header()
hdr["BUNIT"] = "DN/s"
hdr["TGROUP"] = (TGROUP_FULL_FASTR1, "seconds; FULL FASTR1")
hdr["NGROUPS"] = ngroups
hdr["NINTS"] = nints
hdr["METHOD"] = "OLS linear fit over all groups; mean over ints"
hdr["SRCFILE"] = CRDS_DARK.name
fits.PrimaryHDU(rate_2d, header=hdr).writeto(OUT_RATE, overwrite=True)
print("Wrote", OUT_RATE)


fitted rows 0:64
fitted rows 64:128
fitted rows 128:192
fitted rows 192:256
fitted rows 256:320
fitted rows 320:384
fitted rows 384:448
fitted rows 448:512
fitted rows 512:576
fitted rows 576:640
fitted rows 640:704
fitted rows 704:768
fitted rows 768:832
fitted rows 832:896
fitted rows 896:960
fitted rows 960:1024
rate_2d DN/s: min=nan max=nan median=0.108 mean=0.1727
Wrote /Users/eckhartspalding/crds_cache/references/jwst/miri/jwst_miri_dark_0113_rate2d.fits
